# Notebook for preprocessing GFAS fire emission prior

In [50]:
import xarray as xr
import os

# Examine the original data structure

In [44]:
DATA_PATH = "/home/pietaril/Documents/data/GFAS/original_GFAS_data/GFAS_CO2_2018.nc"
ds = xr.open_dataset(DATA_PATH)
ds



<xarray.Dataset>
Dimensions:     (valid_time: 365, latitude: 1800, longitude: 3600)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2018-01-01 2018-01-02 ... 2018-12-31
  * latitude    (latitude) float64 89.95 89.85 89.75 ... -89.75 -89.85 -89.95
  * longitude   (longitude) float64 0.05 0.15 0.25 0.35 ... 359.8 359.9 359.9
Data variables:
    co2fire     (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-01-14T15:04 GRIB to CDM+CF via cfgrib-0.9.1...

# Helper functions to preprocess GFAS-type data

In [51]:



def select_area(aoi, ds):
    # Select the area based on area of interest
    
    area_subset = ds.sel(longitude=slice(aoi[0], aoi[1]),
                    latitude=slice(aoi[2], aoi[3]))
    return area_subset

def preprocess_gfas(aoi, ds):
    # order the latitude in ascending order (it's descending in the original)
    out = ds.sortby("latitude")
    out.latitude.attrs.update({"stored_direction": "increasing"},)


    #select aoi
    out = select_area(aoi, out)

    #drop attributes copied from the initial dataset
    out.attrs = {}
    out.attrs.update({"description": "Preprocessed CO2 fire emissions from GFAS. Units and variable names changed from the original to comply with CIF",
                        "units": "kg m-2 hr-1"})

    #rename variables to match CIF standard
    rename_map = {
        "valid_time" : "time",
        "co2fire": "emi_fire"
    }

    out = out.rename(rename_map)
    out.emi_fire.attrs = {}


    #convert to float64
    out["emi_fire"] = out["emi_fire"].astype("float64")

    #convert units from kg/m2/s to kg/m2/hr
    out["emi_fire"] = out["emi_fire"]*60*60

    return out

def write_to_file(dsout, OUT_PATH):
    # Save the result to a new NetCDF file
    year = dsout["time"].dt.year.values[0]
    # Write the output dataset to a new NetCDF file
    fname = f"fire_prior_GFAS_CO2_{year}.nc"
    dsout.to_netcdf(os.path.join(OUT_PATH,fname), encoding={'time':{'units': "days since 1900-01-01"}})
    


In [52]:
DATA_PATH = "/home/pietaril/Documents/data/GFAS/original_GFAS_data/"
OUT_PATH = "/home/pietaril/Documents/data/GFAS/fire_prior_CIF_FLEXPART_CO2M/"
aoi = [-15, 40, 34, 73]


fnames = [os.path.join(DATA_PATH,f) for f in os.listdir(DATA_PATH) if f.endswith(".nc")]
for fname in fnames:
    ds = xr.open_dataset(fname)
    out = preprocess_gfas(aoi, ds)
    write_to_file(out, OUT_PATH)
    ds.close()







In [49]:
out

<xarray.Dataset>
Dimensions:    (time: 366, latitude: 390, longitude: 400)
Coordinates:
  * time       (time) datetime64[ns] 2020-01-01 2020-01-02 ... 2020-12-31
  * latitude   (latitude) float64 34.05 34.15 34.25 34.35 ... 72.75 72.85 72.95
  * longitude  (longitude) float64 0.05 0.15 0.25 0.35 ... 39.75 39.85 39.95
Data variables:
    emi_fire   (time, latitude, longitude) float64 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes:
    description:  Preprocessed CO2 fire emissions from GFAS. Units and variab...
    units:        kg m-2 hr-1